# 01 · Scale & Estimation — The Arithmetic of Systems

**Part 0 · Foundations** · *Prerequisites: 00* · *Book mapping: DDIA ch. 1; background for
everything else*

Good system designers do arithmetic before architecture. The goal is never precision — it is
**order of magnitude**: within ~3×, does this fit on one machine or a thousand? In RAM or on
disk? In one region or behind a CDN? Most bad designs die at this step, cheaply.

**Learning objectives.** After this notebook you can:

1. Recall the latency hierarchy (cache → RAM → SSD → datacenter network → disk → WAN) and use
   it to predict what a design will cost in time.
2. Run the standard estimation pipeline: users → requests/s → peak → storage → bandwidth.
3. Use **Little's law** to size anything that holds work in flight (threads, connections).
4. Explain with queueing theory why systems melt down *before* 100% utilization, and why you
   should plan for ~60–75%.
5. Fit and interpret scalability curves (Amdahl, Universal Scalability Law): why throughput
   stops growing — and can *fall* — as you add machines.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=1)

## 1. Latency numbers you must know

Every design is a walk through the memory/storage/network hierarchy, and each level is a
different *universe* of speed. The table below (rounded 2020s numbers — hardware moves, ratios
barely do) is the single most reused fact-sheet in this course.

The right-hand column rescales everything so that 1 ns becomes 1 second — the classic trick for
making the gaps visceral.

In [ ]:
def human_time(ns: float) -> str:
    """Render nanoseconds at a human scale (used for the 'x1e9' column)."""
    seconds = ns  # after rescaling 1 ns -> 1 s
    for unit, size in [("yr", 365.25 * 86400), ("day", 86400), ("h", 3600),
                       ("min", 60)]:
        if seconds >= size:
            return f"{seconds / size:.1f} {unit}"
    return f"{seconds:.0f} s"

LATENCY_NS = {
    "L1 cache reference":            1,
    "main memory (RAM) reference":   100,
    "read 1 MB sequentially, RAM":   3_000,
    "NVMe SSD random read":          20_000,
    "read 1 MB sequentially, NVMe":  200_000,
    "round trip inside datacenter":  250_000,
    "HDD seek":                      4_000_000,
    "read 1 MB sequentially, HDD":   6_000_000,
    "round trip same continent":     20_000_000,
    "round trip cross-continent":    140_000_000,
}

table = pd.DataFrame({
    "latency": [f"{v:,} ns" if v < 1e3 else
                f"{v/1e3:,.0f} µs" if v < 1e6 else
                f"{v/1e6:,.0f} ms" for v in LATENCY_NS.values()],
    "vs RAM ref": [f"{v/100:,.0f}×" for v in LATENCY_NS.values()],
    "if 1 ns were 1 s": [human_time(v) for v in LATENCY_NS.values()],
}, index=list(LATENCY_NS))
table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
names = list(LATENCY_NS)[::-1]
vals = [LATENCY_NS[n] for n in names]
ax.barh(names, vals, color="#1565c0")
ax.set_xscale("log")
ax.set_xlabel("nanoseconds (log scale!)")
ax.set_title("Eight orders of magnitude — every design walks this ladder")
plt.show()

# The ratios that drive design decisions, verified:
ssd, ram, dc_rtt, wan_rtt = (LATENCY_NS["NVMe SSD random read"],
                             LATENCY_NS["main memory (RAM) reference"],
                             LATENCY_NS["round trip inside datacenter"],
                             LATENCY_NS["round trip cross-continent"])
assert ssd / ram >= 100, "RAM is ~100-1000x faster than SSD -> caching works"
assert dc_rtt / ssd > 5, "a network hop costs more than a local SSD read -> batch your calls"
assert wan_rtt / dc_rtt > 100, "cross-region is a different universe -> geo-replicate reads"

Design consequences you should be able to derive from this ladder on demand:

- **Caching works** because RAM beats SSD by ~1000× and beats a WAN round trip by ~10⁶×.
- **Batching works** because one datacenter round trip costs as much as ~10 SSD reads: never
  make N network calls where 1 call carrying N items will do (the "N+1 query" bug, quantified).
- **Sequential beats random** on every storage medium — the fact behind logs, LSM-trees
  (nb 03), and columnar scans (nb 03, 14).
- **Geography is destiny**: 140 ms cross-continent is unfixable by any amount of engineering —
  only by moving data closer (CDNs, geo-replication, nb 05).

## 2. The estimation pipeline

The standard pipeline turns a product sentence ("10 million people share photos daily") into
capacity numbers. Two conversion tricks do most of the work:

- **A day is ~10⁵ seconds** (86,400). So "N per day" ≈ N/100,000 per second.
- **Peak ≈ 2–5× average** (diurnal cycles, events). We use 3× as the default.

In [ ]:
SECONDS_PER_DAY = 86_400

def per_second(per_day: float) -> float:
    return per_day / SECONDS_PER_DAY

def human_bytes(n: float) -> str:
    """Human-readable byte counts (decimal units, as capacity planning uses)."""
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if n < 1000:
            return f"{n:,.1f} {unit}"
        n /= 1000
    return f"{n:,.1f} EB"

# --- Worked example: a photo-sharing app ---------------------------------
DAU = 10_000_000                  # daily active users
uploads_per_user, views_per_user = 2, 50
photo_bytes = 500_000             # 500 KB after server-side compression
replication = 3                   # every byte stored 3x for durability (nb 05)
retention_years = 5

write_qps = per_second(DAU * uploads_per_user)
read_qps = per_second(DAU * views_per_user)
peak_read_qps = 3 * read_qps

new_bytes_per_day = DAU * uploads_per_user * photo_bytes
stored = new_bytes_per_day * 365 * retention_years * replication
egress_bps = per_second(DAU * views_per_user * photo_bytes) * 8  # bits/s

print(f"writes: {write_qps:,.0f}/s avg  (peak ~{3*write_qps:,.0f}/s)")
print(f"reads:  {read_qps:,.0f}/s avg  (peak ~{peak_read_qps:,.0f}/s)"
      f"  -> read:write ratio {read_qps/write_qps:.0f}:1")
print(f"new data: {human_bytes(new_bytes_per_day)}/day")
print(f"stored after {retention_years}y (x{replication} replicas): {human_bytes(stored)}")
print(f"avg egress: {egress_bps/1e9:,.1f} Gbit/s (peak ~{3*egress_bps/1e9:,.0f} Gbit/s)")

# The conclusions an estimate exists to produce:
assert read_qps / write_qps >= 20, "read-heavy -> optimize the read path (caches, CDN)"
assert stored > 5e15, "petabytes -> object storage + partitioning, not one database"
assert egress_bps > 10e9, "tens of Gbit/s egress -> serve media from a CDN"

Three sentences of product turned into three architecture decisions (cache/CDN the read path,
partition the storage, keep the database for *metadata* only). That is what estimation is for.
Notice what we did **not** compute: anything to two significant figures.

A useful sanity anchor: **100 million users × 1 KB each = 100 GB — that fits in RAM on one big
machine.** Metadata is almost always smaller than intuition says; *media and events* are what
explode. Estimate before you distribute.

## 3. Little's law — sizing what's in flight

> **Little's law**: in any stable system, the average number of items inside it equals arrival
> rate times average time inside: $L = \lambda W$.

It is distribution-free — no assumptions about arrival patterns or service times — which makes
it the universal sizing tool for anything that *holds work*: threads, DB connections, queue
depth, memory for in-flight requests.

Example: a service takes 5,000 requests/s, each spending 200 ms inside (queueing + work). Then
$L = 5000 \times 0.2 = 1000$ requests are in the building at any moment — so a pool of 100
connections *cannot possibly* keep up, no matter how it is tuned.

Distribution-free claims deserve a stress test. We simulate a queue with bursty (Poisson)
arrivals and noisy service times, then measure both sides of the law independently:

In [ ]:
def simulate_queue(lam: float, mu: float, n_jobs: int = 50_000, seed: int = 2):
    """Single-server FIFO queue. Returns (arrival, start, finish) times.

    lam: arrival rate (jobs/unit time), Poisson process.
    mu:  service rate (jobs/unit time), exponential service times.
    """
    r = np.random.default_rng(seed)
    arrival = r.exponential(1 / lam, n_jobs).cumsum()
    service = r.exponential(1 / mu, n_jobs)
    start = np.empty(n_jobs)
    finish = np.empty(n_jobs)
    prev_finish = 0.0
    for i in range(n_jobs):
        # A job starts when it arrives OR when the server frees up - whichever is later.
        start[i] = max(arrival[i], prev_finish)
        finish[i] = start[i] + service[i]
        prev_finish = finish[i]
    return arrival, start, finish

lam, mu = 8.0, 10.0  # 80% utilization
arrival, start, finish = simulate_queue(lam, mu)

# Side 1: average time in system, W (waiting + being served).
W = float((finish - arrival).mean())
# Side 2: time-average number in system, L - measured directly by sampling
# random instants and counting jobs present. No Little's law used here.
sample_ts = rng.uniform(arrival[1000], arrival[-1000], size=400)
L = float(np.mean([(np.sum((arrival <= t) & (finish > t))) for t in sample_ts]))

print(f"measured L = {L:.2f}   lambda x W = {lam * W:.2f}")
assert abs(L - lam * W) / L < 0.1, "Little's law holds without distributional assumptions"

## 4. Why systems melt before 100% utilization

Utilization $\rho = \lambda/\mu$ is the fraction of time the server is busy. Intuition says
"80% busy = fine, 20% headroom". Queueing theory says otherwise: because arrivals are *bursty*,
queues build up during bursts faster than idle periods drain them, and waiting time grows like
$1/(1-\rho)$. For the simplest model (M/M/1 — Poisson arrivals, exponential service, 1 server)
the average wait in queue is exactly:

$$W_q = \frac{\rho}{\mu(1-\rho)}$$

— which is a **hockey stick**: doubling from ρ=0.45 to ρ=0.9 doesn't double the wait; it
multiplies it by ~11. Let's confirm the formula with the simulator, then look at the tail,
which is even crueler:

In [ ]:
utils = np.array([0.3, 0.5, 0.7, 0.8, 0.9, 0.95])
mu = 10.0  # service rate; mean service time 100 ms if the unit is seconds

measured_mean, measured_p99, analytic = [], [], []
for rho in utils:
    arrival, start, finish = simulate_queue(lam=rho * mu, mu=mu, seed=int(rho * 100))
    wait = start - arrival                      # time spent queueing only
    measured_mean.append(float(wait.mean()))
    measured_p99.append(float(np.percentile(wait, 99)))
    analytic.append(rho / (mu * (1 - rho)))

for rho, m, a in zip(utils, measured_mean, analytic):
    assert abs(m - a) / a < 0.15, f"simulation disagrees with M/M/1 formula at rho={rho}"

# The design rule, quantified: waits explode near saturation.
assert measured_mean[4] / measured_mean[1] > 5, "rho 0.5 -> 0.9 multiplies queueing many-fold"

fig, ax = plt.subplots()
ax.plot(utils, np.array(measured_mean) * 1000, "o-", label="mean wait (simulated)")
ax.plot(utils, np.array(analytic) * 1000, "--", label="mean wait (M/M/1 formula)")
ax.plot(utils, np.array(measured_p99) * 1000, "s-", color="#c62828",
        label="p99 wait (simulated)")
ax.set_xlabel("utilization ρ")
ax.set_ylabel("wait in queue (ms)")
ax.set_title("The hockey stick: queueing delay vs utilization (service time 100 ms)")
ax.legend()
plt.show()

Consequences:

- **Plan for 60–75% utilization.** The last 25% of "capacity" is not capacity; it is latency
  poison. Headroom is a feature you buy for your p99.
- **This is why overload cascades.** A dependency at ρ=0.95 answers slowly → callers hold
  connections longer → *their* L = λW grows (Little's law!) → their pools exhaust → the outage
  spreads. Defenses: timeouts, backpressure, load shedding — recurring topics from nb 08 on.
- **Variance is the enemy.** The tail (p99) sits far above the mean at every utilization.
  Anything that smooths service times (bounding request sizes, splitting huge jobs) buys
  capacity for free.

## 5. Scalability laws — what you buy with more machines

"Just add servers" meets two mathematical taxes:

- **Contention** (σ): the serial fraction — work that happens one-at-a-time (a lock, a leader,
  a coordinator). **Amdahl's law**: speedup on N machines is $S(N) = \frac{1}{\sigma + (1-\sigma)/N}$,
  capped forever at $1/\sigma$.
- **Coherence** (κ): the cost of agreement — cache-line ping-pong, gossip, quorums, group
  membership: pairwise coordination that grows with $N^2$. The **Universal Scalability Law**
  adds this term:

$$X(N) = \frac{N \cdot X(1)}{1 + \sigma(N-1) + \kappa N(N-1)}$$

Amdahl plateaus; USL **peaks and then declines** — beyond the peak, adding machines makes the
system *slower*. The peak sits at $N^\* = \sqrt{(1-\sigma)/\kappa}$.

In practice you never know σ and κ a priori; you *measure* throughput at a few cluster sizes
and fit. Let's do exactly that — generate noisy "benchmark measurements" from a hidden truth,
fit USL, and see how well a handful of points recovers the whole curve:

In [ ]:
def usl(n, x1, sigma, kappa):
    """Universal Scalability Law: throughput at cluster size n."""
    return n * x1 / (1 + sigma * (n - 1) + kappa * n * (n - 1))

TRUE = dict(x1=1000.0, sigma=0.05, kappa=0.001)  # hidden ground truth

# "Run the benchmark" at a few sizes, with 3% measurement noise:
sizes = np.array([1, 2, 4, 8, 16, 24, 32, 48])
measured = usl(sizes, **TRUE) * rng.normal(1, 0.03, len(sizes))

# Fit the three parameters from the noisy measurements:
(x1_f, sigma_f, kappa_f), _ = curve_fit(
    usl, sizes, measured, p0=(measured[0], 0.01, 0.0001),
    bounds=([1, 0, 0], [np.inf, 1, 1]),
)
n_star = np.sqrt((1 - sigma_f) / kappa_f)
print(f"fitted: sigma={sigma_f:.3f} kappa={kappa_f:.4f} -> optimal cluster ~{n_star:.0f} nodes")

assert abs(sigma_f - TRUE["sigma"]) < 0.02, "contention recovered from 8 noisy points"
assert abs(kappa_f - TRUE["kappa"]) < 0.001, "coherence recovered from 8 noisy points"
assert 20 < n_star < 40, "there is a finite optimal cluster size"

n_grid = np.arange(1, 101)
amdahl = usl(n_grid, x1_f, sigma_f, 0)  # kappa=0 -> pure Amdahl
fig, ax = plt.subplots()
ax.plot(n_grid, n_grid * x1_f, ":", color="#9e9e9e", label="linear dream")
ax.plot(n_grid, amdahl, "--", label=f"Amdahl (σ={sigma_f:.2f}): plateaus")
ax.plot(n_grid, usl(n_grid, x1_f, sigma_f, kappa_f), "-",
        label=f"USL (+κ={kappa_f:.3f}): peaks then declines")
ax.scatter(sizes, measured, color="#c62828", zorder=3, label="measurements")
ax.axvline(n_star, color="#c62828", linestyle=":", alpha=0.6)
ax.set_xlabel("nodes N")
ax.set_ylabel("throughput (req/s)")
ax.set_title("What you actually buy with more machines")
ax.legend()
plt.show()

Read the fitted parameters like a diagnosis: high **σ** → find and shard the serial piece (a
single leader taking all writes? nb 05–06); high **κ** → reduce coordination (fewer nodes in
each quorum, less chatty protocols, partition so nodes don't need to agree; nb 06, 09). The
entire discipline of distributed data systems is a campaign against these two parameters.

## 6. Exercises

1. **Tweet-scale arithmetic.** 200M DAU; each posts 0.5 tweets/day (280 bytes + 20% metadata
   overhead) and reads a 100-tweet timeline twice a day. Compute average write QPS, average
   timeline-tweet reads/s, and raw new-tweet storage per year (×3 replication). Which single
   number tells you this system's hard problem is reads, not writes?
2. **Pool sizing.** A service calls its database at 2,400 queries/s; queries average 25 ms.
   Size the connection pool with Little's law, then add headroom for 3× peaks. What happens to
   the needed pool if a slow migration makes queries average 250 ms during one hour?
3. **The retrofit trap.** A team runs its API servers at 90% CPU to "save money." Using the
   M/M/1 curve, estimate how the mean queueing delay compares to running at 60%. Include the
   effect on p99 qualitatively.
4. **USL diagnosis.** A cluster benchmark fits to σ=0.02, κ=0.008. Where is the throughput
   peak? The team proposes going from 10 to 20 nodes — what do you tell them?
5. **Latency budget.** A page must render in 200 ms at p95. The path is: client→edge 20 ms,
   edge→app 5 ms, then the app calls 3 services *sequentially* (each does one DB read ~2 ms
   plus ~8 ms compute), then 25 ms render. Does the budget hold? What single structural change
   buys the most headroom?

### Solutions

**1.** Writes: $200\text{M} \times 0.5 / 86{,}400 \approx 1{,}160$/s — trivial. Timeline reads:
$200\text{M} \times 2 \times 100 / 86{,}400 \approx 463{,}000$ tweets/s delivered — ~400× the
write rate. Storage: $100\text{M} \times 336\ \text{B} \times 365 \times 3 \approx 37$ TB/year
— small! The read:write ratio (~400:1) is the number: this is a *read fan-out* problem, which
is why timelines are precomputed (capstone 23).

**2.** $L = 2400 \times 0.025 = 60$ connections busy on average; ×3 peak → ~180, plus margin →
~200. At 250 ms: $L = 600$ average — the pool (and likely the DB) is far beyond exhausted;
this is how a "slow migration" becomes a full outage. Defenses: timeouts + load shedding.

**3.** Mean wait scales with $\rho/(1-\rho)$: at 0.9 → 9 (in units of service time × ρ...
using $W_q = \frac{\rho}{\mu(1-\rho)}$: $0.9/(1-0.9) = 9$ vs $0.6/(1-0.6) = 1.5$) — **6× the
mean queueing delay**, and the p99 grows even more because the wait distribution's tail
lengthens. The "saved" 30% of CPU was purchased with user latency.

**4.** $N^* = \sqrt{(1-0.02)/0.008} \approx 11$. They are already at the peak; going to 20
nodes will *reduce* throughput. The money should go to cutting κ (less coordination) instead.

**5.** Sequential sum: $20 + 5 + 3 \times (2 + 8) + 25 = 80$ ms of *means* — but p95 budgets
must add queueing and tail effects, and three sequential service calls stack their tails
(nb 00 §3.2). The structural change: make the three service calls **parallel** → path cost
drops from 30 ms to ~max(10) ms and only one tail is stacked... at the price of fan-out
amplification. Trade-offs everywhere; measure both.

In [ ]:
# Spot-checks for the solution arithmetic.
assert abs(200e6 * 0.5 / 86400 - 1157) < 5
assert abs(200e6 * 2 * 100 / 86400 - 462_963) < 1000
assert abs(100e6 * 336 * 365 * 3 / 1e12 - 36.8) < 0.5     # TB/year
assert abs(2400 * 0.025 - 60) < 1e-9
assert abs(np.sqrt((1 - 0.02) / 0.008) - 11.07) < 0.05
assert 20 + 5 + 3 * 10 + 25 == 80
print("solution arithmetic verified")

## 7. Takeaways

- Carry the **latency ladder** in your head; designs are sums of its rungs. Cache because RAM
  ≫ SSD ≫ WAN; batch because a round trip ≫ a read; go sequential because seeks are poison.
- Estimation pipeline: daily → /10⁵ → ×3 peak → × size × retention × replication. Output =
  decisions (what to cache, what to partition, what to CDN), not precise numbers.
- **L = λW** sizes everything that holds work in flight — and explains how slowness turns into
  resource exhaustion.
- Queueing delay ~ $1/(1-\rho)$: the hockey stick. Buy headroom; run at 60–75%.
- **USL**: contention (σ) caps you, coherence (κ) reverses you; measure, fit, and attack the
  dominant parameter.

**Further reading.** DDIA ch. 1; Gunther, *Guerrilla Capacity Planning* (USL); Dean & Barroso,
*The Tail at Scale*; Jeff Dean's "numbers everyone should know" (various talks).

**Next:** Part 1 begins — [02 · Data models](../part1-data-systems/02-data-models.ipynb):
the same application built on three different data models, and how to choose.